# provenance — 이 숫자들은 어디서 나왔나 (읽기 전용)

**아무것도 학습하거나 평가하지 않는다.** 디스크에 있는 체크포인트와 eval 결과를 읽어서
"무엇으로, 몇 step 까지, 어떤 설정으로 만들어졌는지"를 표로 찍는다. 몇 초면 끝난다.

## 왜 필요한가

overnight 결과가 논문 표와 안 맞았다.

| | 논문 표 | overnight (이 서버, 100k 체크포인트) |
|---|---:|---|
| BiMamba K=100 s10 | **65.0** | seed1 **50.8**, seed2 **55.4**, seed0 **셀 없음** |
| ACM2 K=100 s10 | 56.8 | seed0 59.6, seed1 58.6, seed2 54.8 |

그리고 "학습" 단계는 사실 **아무것도 학습하지 않았다** — seed 1·2 체크포인트가 이미 있어서 skip 됐다
(`bimamba_pure` 는 150k, `acm2` 는 200k 까지 학습된 옛 run).

**확인하려는 것**

1. 어떤 (태그, seed) 가 디스크에 있고 각각 몇 step 까지 갔나
2. **seed 별 BiMamba 설정이 같은가** (`use_bimamba_decoder`, carry, chunk pairs, scan 모드, 총 step, lr, 스케줄러)
3. eval 결과 파일은 **언제·몇 ep·어떤 체크포인트로** 만들어졌나 (eval 경로에는 step 이 안 들어간다)
4. 표의 **65.0 이 이 서버의 어느 셀과 대응되나**

⚠️ eval 결과 경로 `eval_clean/<task>/<model>/seed<N>/` 에는 **체크포인트 step 이 안 들어간다.**
그래서 같은 경로의 값이 100k 인지 150k 인지는 파일만 봐서는 모른다. 아래 셀이 로그와 수정 시각으로 추적한다.

## 0) 부팅

In [ ]:
import json, os, sys, time
from pathlib import Path

_h = Path.cwd()
_r = next(c for c in (_h, *_h.parents) if (c / 'notebooks' / 'libero' / 'exp5_tonight.py').exists())
for _p in (_r / 'notebooks', _r / 'notebooks' / 'libero'):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import importlib
import exp5_tonight as X
X = importlib.reload(X)
cf, v23 = X.setup()            # 태그 등록. 학습·eval 은 하지 않는다.

TASK = X.TASK
K = 100
SEEDS = [0, 1, 2, 3]
TAGS = ['bimamba_pure', 'bimamba', 'acm2', 'act']     # bimamba = carry 붙은 BiMOS
print('TASK', TASK, '| K', K, '| seeds', SEEDS)
print('OUTPUT_BASE =', v23.OUTPUT_BASE)

def mtime(p):
    try:
        return time.strftime('%m-%d %H:%M', time.localtime(Path(p).stat().st_mtime))
    except Exception:
        return '--'

def dig(obj, keys):
    """중첩 dict 에서 keys 에 해당하는 값을 재귀로 찾아 {key: value} 로 돌려준다."""
    found = {}
    def walk(o):
        if isinstance(o, dict):
            for k, v in o.items():
                if k in keys and k not in found and not isinstance(v, (dict, list)):
                    found[k] = v
                walk(v)
        elif isinstance(o, list):
            for v in o:
                walk(v)
    walk(obj)
    return found

def load_json(p):
    try:
        return json.loads(Path(p).read_text(encoding='utf-8'))
    except Exception:
        return None

## 1) 체크포인트 — 무엇이 몇 step 까지 있나

`v23.train_dir(tag, seed, task)` 아래 `checkpoints/` 를 읽는다. K=100 태그는 접미사가 없다.

In [ ]:
def _tag(base):
    return X.tag_of({'bimamba_pure': 'bimamba', 'bimamba': 'bimos', 'acm2': 'acm2', 'act': 'act'}[base], K)

print(f"{'tag':<14} {'seed':>4}  {'존재':<4} {'step 목록':<44} {'최신 수정'}")
print('-' * 84)
INV = {}
for base in TAGS:
    try:
        t = _tag(base)
    except Exception:
        t = base
    for s in SEEDS:
        d = v23.train_dir(t, s, TASK)
        ck = d / 'checkpoints'
        steps = sorted(int(x.name) for x in ck.iterdir() if x.name.isdigit()) if ck.is_dir() else []
        INV[(t, s)] = steps
        show = ','.join(f'{x // 1000}k' for x in steps) if steps else '-'
        if len(show) > 44:
            show = show[:20] + ' ... ' + show[-18:]
        print(f"{t:<14} {s:>4}  {'O' if steps else 'X':<4} {show:<44} {mtime(ck) if steps else ''}")
    print()

## 2) 설정 — seed 별 BiMamba 가 같은 모델인가

각 (태그, seed) 의 **가장 큰 step 체크포인트**에서 `config.json` 과 `train_config.json` 을 읽어
비교에 필요한 필드만 뽑는다. **같은 태그인데 seed 마다 값이 다르면 그게 원인 후보다.**

봐야 할 것:
- `use_bimamba_decoder`, `sscp_enabled`(carry), `use_chunk_pairs` — **순수 BiMamba 인가**
- `bimamba_scan`, `bimamba_fuse` — 65.0 을 낸 구성과 같은가 (없으면 옛 코드로 학습된 것)
- `steps`, `lr`, 스케줄러 — **총 step 이 다른 run 의 100k 체크포인트를 비교해도 되는가**

In [ ]:
CFG_KEYS = ['type', 'chunk_size', 'n_action_steps', 'use_bimamba_decoder', 'sscp_enabled',
            'use_chunk_pairs', 'bimamba_scan', 'bimamba_fuse', 'carry_fusion', 'dim_model',
            'n_decoder_layers', 'use_action_self_attention']
TR_KEYS = ['steps', 'seed', 'lr', 'optimizer_lr', 'scheduler', 'batch_size', 'save_freq',
           'use_policy_training_preset']

ROWS = []
for (t, s), steps in INV.items():
    if not steps:
        continue
    last = max(steps)
    pm = v23.train_dir(t, s, TASK) / 'checkpoints' / f'{last:06d}' / 'pretrained_model'
    if not pm.is_dir():
        pm = v23.train_dir(t, s, TASK) / 'checkpoints' / str(last) / 'pretrained_model'
    cfg = load_json(pm / 'config.json') or {}
    trc = load_json(pm / 'train_config.json') or {}
    row = {'tag': t, 'seed': s, 'ckpt_step': last, **dig(cfg, set(CFG_KEYS)), **dig(trc, set(TR_KEYS))}
    ROWS.append(row)

def show_rows(rows, cols):
    print(' | '.join(f'{c:<{max(len(c), 9)}}' for c in cols))
    print('-' * (sum(max(len(c), 9) + 3 for c in cols)))
    for r in rows:
        print(' | '.join(f"{str(r.get(c, '-')):<{max(len(c), 9)}}" for c in cols))

cols = ['tag', 'seed', 'ckpt_step', 'steps', 'use_bimamba_decoder', 'sscp_enabled',
        'use_chunk_pairs', 'bimamba_scan', 'bimamba_fuse', 'lr']
show_rows([r for r in ROWS if r['tag'].startswith(('bimamba', 'acm2'))], cols)

print()
print('같은 태그 안에서 seed 마다 다른 필드 (원인 후보):')
by_tag = {}
for r in ROWS:
    by_tag.setdefault(r['tag'], []).append(r)
any_diff = False
for t, rs in by_tag.items():
    if len(rs) < 2:
        continue
    for k in sorted(set().union(*[set(r) for r in rs]) - {'tag', 'seed'}):
        vals = {str(r.get(k, '-')) for r in rs}
        if len(vals) > 1:
            any_diff = True
            print(f'  {t:<14} {k:<22} ' + ' / '.join(f"seed{r['seed']}={r.get(k, '-')}" for r in rs))
if not any_diff:
    print('  (없다 — seed 별 설정은 동일하다)')

## 3) eval 결과 — 언제·몇 ep·어떤 체크포인트로 만들어졌나

`rm_<variant>_k100_s10` 셀(K=100, 실행 stride 10)을 seed 별로 읽는다.
`pc_success` 가 논문 표의 성공률이고 `n_episodes` 가 500 이어야 표와 같은 조건이다.

**수정 시각**이 중요하다. overnight 가 오늘 새로 만든 결과(seed 1·2)와 그 전에 있던 결과(seed 0)를 가른다.

In [ ]:
VARS = ['act', 'acm2', 'bimamba', 'bimos', 'carry', 'bimamba_cpoff']
print(f"{'cell':<28} {'seed':>4}  {'pc_success':>10} {'n_ep':>5}  {'수정 시각'}")
print('-' * 66)
EVAL = {}
for var in VARS:
    out = f'rm_{var}_k{K}_s{X.MAIN_STRIDE}'
    if out not in v23.MODEL_DIR_NAMES:
        continue
    for s in SEEDS:
        info = v23.eval_clean_dir(out, s, TASK) / 'eval_info.json'
        if not info.exists():
            print(f"{out:<28} {s:>4}  {'--':>10} {'--':>5}")
            continue
        d = load_json(info) or {}
        ov = d.get('overall', {}) if isinstance(d, dict) else {}
        n = ov.get('n_episodes', ov.get('n_ep'))
        EVAL[(out, s)] = {'sr': ov.get('pc_success'), 'n': n, 'mtime': mtime(info), 'path': str(info)}
        print(f"{out:<28} {s:>4}  {ov.get('pc_success', float('nan')):>10.1f} {str(n):>5}  {mtime(info)}")
    print()

## 4) 그 eval 이 실제로 연 체크포인트

eval 로그(`_logs/exp5__<cell>.log`)의 앞부분에 실행 커맨드가 남는다. 거기서 `checkpoints/<step>` 을
찾아 **정말 100k 로 평가됐는지** 확인한다.

⚠️ 로그 이름이 셀 이름만으로 정해져서 **seed 를 바꿔 돌리면 덮어쓴다.** 마지막에 돈 seed 것만 남는다.

In [ ]:
import re

LOGDIR = v23.OUTPUT_BASE / '_logs'
print('로그 폴더:', LOGDIR, '(있음)' if LOGDIR.is_dir() else '(없음)')
for var in ['bimamba', 'acm2']:
    out = f'rm_{var}_k{K}_s{X.MAIN_STRIDE}'
    lg = LOGDIR / f'exp5__{out}.log'
    print(f'\n[{out}]  {lg.name}  수정 {mtime(lg)}')
    if not lg.exists():
        print('   로그 없음')
        continue
    txt = lg.read_text(encoding='utf-8', errors='replace')
    head = txt[:6000]
    hits = re.findall(r'[^\s\'\"]*checkpoints/[0-9]+[^\s\'\"]*', head)
    seeds = re.findall(r'seed[0-9]+', head)
    print('   체크포인트 경로:', sorted(set(hits))[:4] or '(로그 앞부분에 없음)')
    print('   seed 표기      :', sorted(set(seeds))[:6])

## 5) 표의 65.0 은 어디에 있나 — 이 서버 전체 검색

`eval_info.json` 중 `pc_success` 가 **64.5~65.5** 인 것을 전부 찾는다. 같은 조건(500 ep)의 다른 셀이
있으면 그게 논문 표의 출처일 수 있다. 없으면 **은지님 쪽 서버/브랜치에서 나온 값**이다.

In [ ]:
hits = []
root = v23.OUTPUT_BASE / 'eval_clean'
for p in root.rglob('eval_info.json'):
    d = load_json(p)
    if not isinstance(d, dict):
        continue
    ov = d.get('overall', {})
    sr = ov.get('pc_success')
    if sr is not None and 64.5 <= float(sr) <= 65.5:
        hits.append((str(p.relative_to(root)), sr, ov.get('n_episodes'), mtime(p)))
print(f'pc_success 64.5~65.5 인 eval: {len(hits)} 개')
for h in sorted(hits):
    print(f'  {h[1]:>5.1f}  n={h[2]}  {h[3]}  {h[0]}')
if not hits:
    print('  (없다 — 표의 65.0 은 이 서버의 eval_clean 에 없다)')

## 6) 읽는 법

| 셀 | 보는 것 | 뜻 |
|---|---|---|
| §2 | seed 마다 **다른 필드** | 있으면 그게 seed 간 BiMamba 성능 차이의 후보 |
| §2 | `steps` 가 태그마다 다름 (150k vs 200k) | 그 run 들의 100k 체크포인트를 같은 조건으로 봐도 되는지 별도 확인 필요 |
| §3 | seed 0 BiMamba 가 `--` | 이 서버에는 seed 0 BiMamba K=100 s10 결과가 **없다** |
| §3 | `n_ep` 가 500 이 아님 | 표와 조건이 다르다 |
| §4 | 체크포인트 경로의 step | 100000 이 아니면 그 결과는 100k 가 아니다 |
| §5 | 65 근처 결과 | 있으면 대응 셀 확인. 없으면 은지님 서버 값 |

### 은지님께 물을 것 (이 노트북 결과를 들고)

1. 표의 **BiMamba 65.0** 은 어떤 체크포인트(브랜치·config·step·seed)로 어느 서버에서 쟀나
2. 같은 조건의 **ACM2 56.8** 은?
3. 이 서버의 `bimamba_pure` seed 1·2 는 그 65.0 과 같은 구성인가

이 셋이 답해지기 전에는 **gap 을 "재현됐다/안 됐다" 어느 쪽으로도 단정하지 않는다.**